# Supervised Trajectories — Reject and Revise, and Abort

This notebook shows two trajectories available with `SupervisedTaskHandler`
that are not possible with `run()` or `run(with_approval=True)`:

- **Reject and Revise.** After the agent delivers its result, the human calls
  `reject()` with structured feedback. The handler routes the rejection back
  through `get_next_step()` deterministically — the agent revises without
  replanning from scratch.
- **Abort.** The human calls `abort()` mid-execution to terminate cleanly,
  setting a `TaskHandlerError` (or a custom exception) on the handler that
  the caller can inspect.

Both trajectories use a code refactoring scenario to keep the focus on the
HITL mechanics rather than tooling.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama
installed on your local machine and have its LLM hosting service running.
To download Ollama, follow the instructions found on this page:
https://ollama.com/download. After downloading and installing Ollama, you
can start a service by opening a terminal and running `ollama serve`.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

In [ ]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm)

## Trajectory 1: Reject and Revise

The agent is asked to refactor a function by replacing its `if-elif` chain
with a dictionary dispatch. After it delivers its result, the human inspects
it, finds it incomplete (no error handling), and calls `reject()` with
specific feedback. The handler feeds the rejection back through
`get_next_step()` and the agent revises.

In [ ]:
ORIGINAL = """\
def calculate(x, y, operation):
    if operation == \"add\":
        return x + y
    elif operation == \"subtract\":
        return x - y
    elif operation == \"multiply\":
        return x * y
    elif operation == \"divide\":
        return x / y
"""

In [ ]:
task = Task(
    instruction=(
        "Refactor the following Python function to replace the "
        f"if-elif chain with a dictionary dispatch:\n\n{ORIGINAL}"
    ),
)
task_handler = await agent.run_supervised(task)

In [ ]:
step = await task_handler.get_next_step(None)
step

In [ ]:
step_result = await task_handler.run_step(step)
step_result

In [ ]:
result = await task_handler.get_next_step(step_result)
result

In [ ]:
rejected = task_handler.reject(
    result,
    feedback=(
        "Good start, but add error handling: raise ValueError for "
        "unknown operations and ZeroDivisionError for division by zero."
    ),
)
rejected

In [ ]:
step = await task_handler.get_next_step(rejected)
step

In [ ]:
step_result = await task_handler.run_step(step)
step_result

In [ ]:
result = await task_handler.get_next_step(step_result)
result

In [ ]:
await task_handler.complete(result)
print(result.content)

## Trajectory 2: Abort

The agent is asked to refactor the authentication module. After seeing the
first planned step, the human realises the task conflicts with an ongoing
security audit and calls `abort()`. The handler terminates cleanly and the
caller can inspect the exception that was set.

In [ ]:
task2 = Task(
    instruction=(
        "Refactor the authentication module to consolidate all login, "
        "logout, and session renewal functions into a single AuthService class."
    ),
)
task_handler2 = await agent.run_supervised(task2)

In [ ]:
step = await task_handler2.get_next_step(None)
step

In [ ]:
await task_handler2.abort(
    error=RuntimeError(
        "Refactoring auth is blocked pending the security audit. "
        "Revisit after the audit completes.",
    ),
)

In [ ]:
task_handler2.exception()